# 01. Data Preparation

## 1. Data Audit & Cleaning

In [2]:
from pathlib import Path 
import pandas as pd 


In [3]:
data_path = Path("data/raw/DILIrank2_Dataset_FDA.xlsx")
data = pd.read_excel(data_path, header=1, dtype=str)


In [4]:
display(data.head(5))
print(data.shape)


,LTKBID,CompoundName,SeverityClass,LabelSection,vDILI-Concern,Comment
0,LT00040,Abacavir sulfate,8,Warnings & precautions,vMOST-DILI-concern,Unchanged
1,LT03618,Abaloparatide,0,No match,vNo-DILI-concern,New
2,LT01402,Abatacept,0,No match,vLess-DILI-concern,Unchanged
3,LT01330,Abciximab,0,No match,vNo-DILI-concern,Unchanged
4,LT03619,Abemaciclib,3,Warnings & precautions,vLess-DILI-concern,New


(1336, 6)


In [5]:
data.columns.tolist()

['LTKBID',
 'CompoundName',
 'SeverityClass',
 'LabelSection',
 'vDILI-Concern',
 'Comment']

In [6]:
# look for unique values in the following columns
cols = ["SeverityClass", "LabelSection", "vDILI-Concern", "Comment"]

for col in cols:
    print(f"Column: {col}")
    print(data[col].unique())
    print("\n")


Column: SeverityClass
<StringArray>
['8', '0', '3', '4', '5', '7', '2', '6', '1']
Length: 9, dtype: str


Column: LabelSection
<StringArray>
['Warnings & precautions',               'No match',      'Adverse reactions',
            'Box warning',              'Withdrawn',           'Discontinued']
Length: 6, dtype: str


Column: vDILI-Concern
<StringArray>
[    'vMOST-DILI-concern',       'vNo-DILI-concern',     'vLess-DILI-concern',
     'vMost-DILI-concern', 'Ambiguous-DILI-concern',       'vNo-DILI-Concern']
Length: 6, dtype: str


Column: Comment
<StringArray>
['Unchanged', 'New', 'Revised']
Length: 3, dtype: str




In [7]:
# normalize DILI concern labels
concern_mapping = {
    "vMOST-DILI-concern": "vMost-DILI-concern",
    "vMost-DILI-concern": "vMost-DILI-concern",
    "vLess-DILI-concern": "vLess-DILI-concern",
    "vNo-DILI-concern": "vNo-DILI-concern",
    "vNo-DILI-Concern": "vNo-DILI-concern",
    "Ambiguous-DILI-concern": "Ambiguous-DILI-concern"
}

data["vDILI-Concern"] = data["vDILI-Concern"].map(concern_mapping)
data["vDILI-Concern"].value_counts(dropna=False)


vDILI-Concern
vNo-DILI-concern          414
Ambiguous-DILI-concern    354
vLess-DILI-concern        351
vMost-DILI-concern        217
Name: count, dtype: int64

In [8]:
# check for rows with missing values
data.isnull().sum()


LTKBID           0
CompoundName     0
SeverityClass    0
LabelSection     0
vDILI-Concern    0
Comment          0
dtype: int64

In [9]:
# check for duplicates in the dataset

# duplicate IDs
print("Duplicate LTKBID:", data["LTKBID"].duplicated().sum())

# duplicate compound names
print("Duplicate compound names:", data["CompoundName"].duplicated().sum())


Duplicate LTKBID: 0
Duplicate compound names: 0


In [10]:
# display the first 5 rows of the dataset
display(data.head(5))

,LTKBID,CompoundName,SeverityClass,LabelSection,vDILI-Concern,Comment
0,LT00040,Abacavir sulfate,8,Warnings & precautions,vMost-DILI-concern,Unchanged
1,LT03618,Abaloparatide,0,No match,vNo-DILI-concern,New
2,LT01402,Abatacept,0,No match,vLess-DILI-concern,Unchanged
3,LT01330,Abciximab,0,No match,vNo-DILI-concern,Unchanged
4,LT03619,Abemaciclib,3,Warnings & precautions,vLess-DILI-concern,New


In [11]:
# data summary
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1336 entries, 0 to 1335
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   LTKBID         1336 non-null   str  
 1   CompoundName   1336 non-null   str  
 2   SeverityClass  1336 non-null   str  
 3   LabelSection   1336 non-null   str  
 4   vDILI-Concern  1336 non-null   str  
 5   Comment        1336 non-null   str  
dtypes: str(6)
memory usage: 62.8 KB


## 2. Molecular Structure Retrieval

We keep this step simple:

1. query PubChem once using the DILIrank `CompoundName` exactly as provided;
2. keep successful matches;
3. leave `not_found` and `multiple_candidates` unresolved for now;
4. validate only the successfully retrieved SMILES with RDKit.

No synonym search, autocomplete, fuzzy matching, or alternative-name recovery is performed in this notebook.

### 2.1 PubChem setup

In [12]:
import time
import requests

from urllib.parse import quote
from tqdm.auto import tqdm

/home/gman/Documents/BS-Bioinformatics/dilirank2-prediction/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
INTERIM_DIR = Path("data/interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# New clean file for the simple exact-name lookup.
# This does not use the previous automated candidate-resolution file.
INTERIM_PATH = INTERIM_DIR / "dilirank2_structures.csv"

BASE_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

session = requests.Session()
session.headers.update({
    "User-Agent": "DILIrank2-benchmark/1.0"
})

print(f"Exact PubChem lookup file: {INTERIM_PATH}")

Exact PubChem lookup file: data/interim/dilirank2_structures.csv


### 2.2 Create or load the exact-lookup table

The interim table always contains exactly one row per DILIrank record.

On the first run, all 1,336 compounds start as `pending`. On later runs, already completed exact-name lookups are reused.

In [14]:
lookup_columns = [
    "PubChemCID",
    "PubChemTitle",
    "SMILES_raw",
    "InChIKey_raw",
    "pubchem_status",
    "pubchem_candidate_cids",
    "rdkit_valid",
    "SMILES_rdkit",
]

if INTERIM_PATH.exists():
    structures = pd.read_csv(
        INTERIM_PATH,
        dtype={"LTKBID": str}
    )
else:
    structures = data[["LTKBID", "CompoundName"]].copy()

    for column in lookup_columns:
        structures[column] = pd.NA

    structures["pubchem_status"] = "pending"
    structures.to_csv(INTERIM_PATH, index=False)

for column in lookup_columns:
    if column not in structures.columns:
        structures[column] = pd.NA

if len(structures) != len(data) or structures["LTKBID"].nunique() != len(data):
    raise ValueError(
        "The interim lookup table should contain exactly one row per DILIrank record."
    )

print(f"Rows: {len(structures)}")
display(structures["pubchem_status"].value_counts(dropna=False).to_frame("count"))

Rows: 1336


,count
pubchem_status,
found,1215
not_found,113
multiple_candidates,8


### 2.3 Exact PubChem name lookup

In [15]:
def fetch_pubchem_structure(name, max_retries=4):
    """Retrieve PubChem structure information using the exact DILIrank name."""

    encoded_name = quote(str(name).strip(), safe="")
    properties = "Title,SMILES,InChIKey"

    url = (
        f"{BASE_URL}/compound/name/{encoded_name}"
        f"/property/{properties}/JSON"
    )

    for attempt in range(max_retries):
        try:
            response = session.get(url, timeout=30)

            if response.status_code == 404:
                return {
                    "pubchem_status": "not_found",
                    "PubChemCID": None,
                    "PubChemTitle": None,
                    "SMILES_raw": None,
                    "InChIKey_raw": None,
                    "pubchem_candidate_cids": None,
                }

            if response.status_code in {429, 500, 502, 503, 504}:
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue

            response.raise_for_status()

            compounds = (
                response.json()
                .get("PropertyTable", {})
                .get("Properties", [])
            )

            if len(compounds) == 0:
                return {
                    "pubchem_status": "not_found",
                    "PubChemCID": None,
                    "PubChemTitle": None,
                    "SMILES_raw": None,
                    "InChIKey_raw": None,
                    "pubchem_candidate_cids": None,
                }

            if len(compounds) > 1:
                candidate_cids = ";".join(
                    str(compound.get("CID"))
                    for compound in compounds
                )

                return {
                    "pubchem_status": "multiple_candidates",
                    "PubChemCID": None,
                    "PubChemTitle": None,
                    "SMILES_raw": None,
                    "InChIKey_raw": None,
                    "pubchem_candidate_cids": candidate_cids,
                }

            compound = compounds[0]

            return {
                "pubchem_status": "found",
                "PubChemCID": compound.get("CID"),
                "PubChemTitle": compound.get("Title"),
                "SMILES_raw": compound.get("SMILES"),
                "InChIKey_raw": compound.get("InChIKey"),
                "pubchem_candidate_cids": str(compound.get("CID")),
            }

        except requests.RequestException:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue

            return {
                "pubchem_status": "request_error",
                "PubChemCID": None,
                "PubChemTitle": None,
                "SMILES_raw": None,
                "InChIKey_raw": None,
                "pubchem_candidate_cids": None,
            }

### 2.4 Retrieve structures

Only `pending` rows and temporary `request_error` rows are queried.

A compound already marked `not_found` is **not searched again**.

In [16]:
to_retrieve = structures.loc[
    structures["pubchem_status"].isin(["pending", "request_error"]),
    ["LTKBID", "CompoundName"]
].copy()

print(f"Compounds to query: {len(to_retrieve)}")

Compounds to query: 0


In [17]:
for i, row in enumerate(
    tqdm(
        to_retrieve.itertuples(index=False),
        total=len(to_retrieve),
        desc="Querying PubChem"
    ),
    start=1
):
    result = fetch_pubchem_structure(row.CompoundName)
    mask = structures["LTKBID"].eq(row.LTKBID)

    for column, value in result.items():
        structures.loc[mask, column] = value

    if i % 25 == 0:
        structures.to_csv(INTERIM_PATH, index=False)

    time.sleep(0.25)

structures.to_csv(INTERIM_PATH, index=False)

display(
    structures["pubchem_status"]
    .value_counts(dropna=False)
    .to_frame("count")
)

Querying PubChem: 0it [00:00, ?it/s]


,count
pubchem_status,
found,1215
not_found,113
multiple_candidates,8


### 2.5 Inspect PubChem retrieval results

In [18]:
not_found = structures.loc[
    structures["pubchem_status"] == "not_found",
    ["LTKBID", "CompoundName"]
].copy()

multiple_candidates = structures.loc[
    structures["pubchem_status"] == "multiple_candidates",
    ["LTKBID", "CompoundName", "pubchem_candidate_cids"]
].copy()

request_errors = structures.loc[
    structures["pubchem_status"] == "request_error",
    ["LTKBID", "CompoundName"]
].copy()

print(f"Found: {(structures['pubchem_status'] == 'found').sum()}")
print(f"Not found: {len(not_found)}")
print(f"Multiple candidates: {len(multiple_candidates)}")
print(f"Request errors: {len(request_errors)}")

print()
print("Not found:")
display(not_found)

print()
print("Multiple candidates:")
display(multiple_candidates)

Found: 1215
Not found: 113
Multiple candidates: 8
Request errors: 0

Not found:


,LTKBID,CompoundName
2,LT01402,Abatacept
3,LT01330,Abciximab
18,LT01387,Adalimumab
22,LT02385,Agalsidase beta
28,LT01319,Aldesleukin
...,...,...
1251,LT01352,Trastuzumab
1283,LT01229,Urokinase
1285,LT01416,Ustekinumab
1302,LT01075,Verteporfin



Multiple candidates:


,LTKBID,CompoundName,pubchem_candidate_cids
116,LT00512,Balsalazide disodium,135413496;135414247;54585;135515521;136040192
135,LT01123,Benztropine mesylate,238053;3246155;8584
381,LT00986,Dolasetron mesylate,6918119;3033817;22058225;60653;24839546;706873...
581,LT00557,Granisetron hydrochloride,6918003;65264;49799997
877,LT00602,Nystatin,6433272;11286230;4568;5702099;16219709;2484066...
1109,LT01974,Scopolamine,3000322;5184;11968014;153311;6451257;452977;27...
1230,LT01037,Tirofiban hydrochloride,64781;60946
1272,LT00516,Trospium chloride,5284631;5284632;123134378


## 3. RDKit Structure Validation

Now validate only the structures that PubChem resolved successfully.

`Chem.MolFromSmiles()` parses the SMILES and performs RDKit's normal molecule sanitization. If it returns a molecule, we mark the structure as valid. We also save RDKit's canonical isomeric SMILES as a reproducible parsed representation.

This is **validation only**. We are not yet standardizing salts, neutralizing structures, or changing tautomers.

In [19]:
from rdkit import Chem

In [20]:
def validate_smiles(smiles):
    """Check whether RDKit can parse and sanitize a SMILES string."""

    if pd.isna(smiles) or not str(smiles).strip():
        return False, None

    try:
        mol = Chem.MolFromSmiles(str(smiles))

        if mol is None:
            return False, None

        rdkit_smiles = Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=True
        )

        return True, rdkit_smiles

    except Exception:
        return False, None

In [21]:
found_rows = structures.loc[
    structures["pubchem_status"] == "found",
    ["LTKBID", "SMILES_raw"]
].copy()

for row in tqdm(
    found_rows.itertuples(index=False),
    total=len(found_rows),
    desc="Validating SMILES with RDKit"
):
    is_valid, rdkit_smiles = validate_smiles(row.SMILES_raw)

    mask = structures["LTKBID"].eq(row.LTKBID)
    structures.loc[mask, "rdkit_valid"] = is_valid
    structures.loc[mask, "SMILES_rdkit"] = rdkit_smiles

structures.to_csv(INTERIM_PATH, index=False)

Validating SMILES with RDKit: 100%|██████████| 1215/1215 [00:00<00:00, 1454.61it/s]


### 3.1 Audit RDKit validation

In [33]:
found_mask = structures["pubchem_status"].eq("found")
valid_mask = structures["rdkit_valid"].fillna(False).astype(bool)

n_found = found_mask.sum()
n_valid = (found_mask & valid_mask).sum()
n_invalid = (found_mask & ~valid_mask).sum()

print(f"PubChem structures found: {n_found}")
print(f"RDKit-valid structures: {n_valid}")
print(f"RDKit-invalid structures: {n_invalid}")

invalid_structures = structures.loc[
    found_mask & ~valid_mask,
    [
        "LTKBID",
        "CompoundName",
        "PubChemCID",
        "SMILES_raw",
    ]
].copy()


PubChem structures found: 1215
RDKit-valid structures: 1215
RDKit-invalid structures: 0


### 3.2 Attach retrieved and validated structures to the working dataset

In [23]:
columns_to_attach = [
    "LTKBID",
    "PubChemCID",
    "PubChemTitle",
    "SMILES_raw",
    "InChIKey_raw",
    "pubchem_status",
    "pubchem_candidate_cids",
    "rdkit_valid",
    "SMILES_rdkit",
]

data = data.merge(
    structures[columns_to_attach],
    on="LTKBID",
    how="left",
    validate="one_to_one"
)

display(
    data[
        [
            "LTKBID",
            "CompoundName",
            "vDILI-Concern",
            "PubChemCID",
            "SMILES_raw",
            "pubchem_status",
            "rdkit_valid",
        ]
    ].head()
)

,LTKBID,CompoundName,vDILI-Concern,PubChemCID,SMILES_raw,pubchem_status,rdkit_valid
0,LT00040,Abacavir sulfate,vMost-DILI-concern,441384.0,C1CC1NC2=C3C(=NC(=N2)N)N(C=N3)[C@@H]4C[C@@H](C...,found,True
1,LT03618,Abaloparatide,vNo-DILI-concern,76943386.0,CC[C@H](C)[C@@H](C(=O)N[C@@H](CCC(=O)N)C(=O)N[...,found,True
2,LT01402,Abatacept,vLess-DILI-concern,NaN,NaN,not_found,NaN
3,LT01330,Abciximab,vNo-DILI-concern,NaN,NaN,not_found,NaN
4,LT03619,Abemaciclib,vLess-DILI-concern,46220502.0,CCN1CCN(CC1)CC2=CN=C(C=C2)NC3=NC=C(C(=N3)C4=CC...,found,True


## 4. Structure Composition Audit

At this point, PubChem resolved 1,215 structures and all 1,215 were successfully parsed by RDKit.

From here onward, this notebook works only with those successfully retrieved and RDKit-valid structures. **Ambiguous DILI-concern compounds are retained**; target-specific filtering belongs in the modelling workflow, not in data preparation.

Before standardization, we record the number of disconnected fragments in each molecule. This identifies salts, counterions, and other multi-fragment structures without removing them.

In [24]:
# Keep only successfully retrieved, RDKit-valid structures.
found_mask = data["pubchem_status"].eq("found")
valid_mask = data["rdkit_valid"].fillna(False).astype(bool)

prepared = data.loc[found_mask & valid_mask].copy().reset_index(drop=True)

print(f"Prepared structures: {len(prepared)}")
print()
display(
    prepared["vDILI-Concern"]
    .value_counts(dropna=False)
    .to_frame("count")
)

assert len(prepared) == n_valid
assert prepared["LTKBID"].nunique() == len(prepared)


Prepared structures: 1215



,count
vDILI-Concern,
vNo-DILI-concern,350
vLess-DILI-concern,330
Ambiguous-DILI-concern,329
vMost-DILI-concern,206


In [25]:
def count_fragments(smiles):
    """Return the number of disconnected molecular fragments in a SMILES string."""

    if pd.isna(smiles) or not str(smiles).strip():
        return pd.NA

    mol = Chem.MolFromSmiles(str(smiles))

    if mol is None:
        return pd.NA

    return len(Chem.GetMolFrags(mol))


prepared["n_fragments"] = prepared["SMILES_rdkit"].apply(count_fragments)

fragment_counts = (
    prepared["n_fragments"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("count")
)

display(fragment_counts)

multi_fragment = prepared.loc[
    prepared["n_fragments"].fillna(0).astype(int) > 1,
    [
        "LTKBID",
        "CompoundName",
        "vDILI-Concern",
        "SMILES_rdkit",
        "n_fragments",
    ]
].copy()

print(f"Multi-fragment structures: {len(multi_fragment)}")
display(multi_fragment)


[09:21:02] WARNING: not removing hydrogen atom without neighbors


,count
n_fragments,
1,655
2,408
3,111
4,20
5,9
6,3
7,3
8,1
9,3


Multi-fragment structures: 560


,LTKBID,CompoundName,vDILI-Concern,SMILES_rdkit,n_fragments
0,LT00040,Abacavir sulfate,vMost-DILI-concern,Nc1nc(NC2CC2)c2ncn([C@H]3C=C[C@@H](CO)C3)c2n1....,3
5,LT01141,Acamprosate calcium,vLess-DILI-concern,CC(=O)NCCCS(=O)(=O)[O-].CC(=O)NCCCS(=O)(=O)[O-...,3
7,LT01035,Acebutolol hydrochloride,vLess-DILI-concern,CCCC(=O)Nc1ccc(OCC(O)CNC(C)C)c(C(C)=O)c1.Cl,2
12,LT02096,Acetylcholine chloride,vNo-DILI-concern,CC(=O)OCC[N+](C)(C)C.[Cl-],2
18,LT02889,Afatinib dimaleate,Ambiguous-DILI-concern,CN(C)C/C=C/C(=O)Nc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2...,3
...,...,...,...,...,...
1195,LT03790,Vorapaxar sulfate,vNo-DILI-concern,CCOC(=O)N[C@@H]1CC[C@@H]2[C@@H](C1)C[C@H]1C(=O...,2
1198,LT02902,Vortioxetine hydrobromide,vLess-DILI-concern,Br.Cc1ccc(Sc2ccccc2N2CCNCC2)c(C)c1,2
1201,LT00647,Warfarin sodium,vLess-DILI-concern,CC(=O)CC(c1ccccc1)c1c([O-])c2ccccc2oc1=O.[Na+],2
1210,LT01017,Ziprasidone hydrochloride,vLess-DILI-concern,Cl.O=C1Cc2cc(CCN3CCN(c4nsc5ccccc45)CC3)c(Cl)cc2N1,2


## 5. Molecular Structure Standardization

We retain three structure levels:

- `SMILES_raw`: the structure returned by PubChem;
- `SMILES_rdkit`: RDKit's canonical isomeric representation of that retrieved structure;
- `SMILES_standardized`: RDKit-cleaned **reported chemical form**, preserving disconnected salt/counterion fragments;
- `SMILES_parent`: RDKit fragment parent, used mainly to detect compounds that collapse to the same parent structure.

The standardized reported form is the structure intended for the main feature-generation workflow. The parent form is retained as an audit/grouping variable and is **not used to silently replace salt-specific DILIrank entries**.

In [26]:
from rdkit.Chem.MolStandardize import rdMolStandardize


In [27]:
def standardize_structure(smiles):
    """Create cleaned reported-form and fragment-parent representations."""

    result = {
        "standardization_ok": False,
        "standardization_error": pd.NA,
        "SMILES_standardized": pd.NA,
        "InChIKey_standardized": pd.NA,
        "SMILES_parent": pd.NA,
        "InChIKey_parent": pd.NA,
    }

    if pd.isna(smiles) or not str(smiles).strip():
        result["standardization_error"] = "missing_smiles"
        return result

    try:
        mol = Chem.MolFromSmiles(str(smiles))

        if mol is None:
            result["standardization_error"] = "rdkit_parse_failed"
            return result

        # Standardize the reported chemical form without discarding fragments.
        standardized = rdMolStandardize.Cleanup(mol)

        result["SMILES_standardized"] = Chem.MolToSmiles(
            standardized,
            canonical=True,
            isomericSmiles=True,
        )
        result["InChIKey_standardized"] = Chem.MolToInchiKey(standardized)

        # Generate a fragment parent separately for grouping/leakage checks.
        parent = rdMolStandardize.FragmentParent(mol)

        result["SMILES_parent"] = Chem.MolToSmiles(
            parent,
            canonical=True,
            isomericSmiles=True,
        )
        result["InChIKey_parent"] = Chem.MolToInchiKey(parent)

        result["standardization_ok"] = True
        return result

    except Exception as exc:
        result["standardization_error"] = f"{type(exc).__name__}: {exc}"
        return result


In [34]:
standardization_records = []

for smiles in tqdm(
    prepared["SMILES_rdkit"],
    total=len(prepared),
    desc="Standardizing structures",
):
    standardization_records.append(
        standardize_structure(smiles)
    )

standardized = pd.DataFrame(
    standardization_records,
    index=prepared.index,
)

for column in standardized.columns:
    prepared[column] = standardized[column]

print(
    prepared["standardization_ok"]
    .value_counts(dropna=False)
    .rename_axis("standardization_ok")
    .to_frame("count")
)

standardization_failures = prepared.loc[
    ~prepared["standardization_ok"].fillna(False),
    [
        "LTKBID",
        "CompoundName",
        "SMILES_rdkit",
        "standardization_error",
    ]
].copy()

print(f"Standardization failures: {len(standardization_failures)}")
# display(standardization_failures)


Standardizing structures:   0%|          | 0/1215 [00:00<?, ?it/s][09:33:56] Initializing MetalDisconnector
[09:33:56] Running MetalDisconnector
[09:33:56] Initializing Normalizer
[09:33:56] Running Normalizer
[09:33:56] Initializing MetalDisconnector
[09:33:56] Running MetalDisconnector
[09:33:56] Initializing Normalizer
[09:33:56] Running Normalizer
[09:33:56] Running LargestFragmentChooser
[09:33:56] Fragment: Nc1nc(NC2CC2)c2ncn([C@H]3C=C[C@@H](CO)C3)c2n1
[09:33:56] New largest fragment: Nc1nc(NC2CC2)c2ncn([C@H]3C=C[C@@H](CO)C3)c2n1 (39)
[09:33:56] Fragment: Nc1nc(NC2CC2)c2ncn([C@H]3C=C[C@@H](CO)C3)c2n1
[09:33:56] New largest fragment: Nc1nc(NC2CC2)c2ncn([C@H]3C=C[C@@H](CO)C3)c2n1 (39)
[09:33:56] Fragment: O=S(=O)(O)O
[09:33:56] Initializing MetalDisconnector
[09:33:56] Running MetalDisconnector
[09:33:56] Initializing Normalizer
[09:33:56] Running Normalizer
[09:33:56] Initializing MetalDisconnector
[09:33:56] Running MetalDisconnector
[09:33:56] Initializing Normalizer
[09:33:56] 

                    count
standardization_ok       
True                 1215
Standardization failures: 0


## 6. Structural Duplicate and Parent-Structure Audit

The original dataset has no duplicate `LTKBID` values or compound names, but different DILIrank records can still share the same chemical structure.

We therefore check two kinds of duplication:

1. **standardized-form duplicates** — identical standardized reported structures;
2. **parent-form duplicates** — records that become identical after salt/counterion removal.

No records are removed here. These flags are retained so chemically equivalent or closely related forms can be handled safely during split design.

In [29]:
# Group sizes for exact standardized forms.
prepared["standardized_group_size"] = (
    prepared.groupby("InChIKey_standardized", dropna=False)["LTKBID"]
    .transform("size")
    .astype(int)
)

# Group sizes after reducing structures to their fragment parent.
prepared["parent_group_size"] = (
    prepared.groupby("InChIKey_parent", dropna=False)["LTKBID"]
    .transform("size")
    .astype(int)
)

prepared["has_standardized_duplicate"] = (
    prepared["standardized_group_size"] > 1
)

prepared["has_parent_duplicate"] = (
    prepared["parent_group_size"] > 1
)

print(
    "Rows belonging to duplicated standardized structures:",
    int(prepared["has_standardized_duplicate"].sum())
)
print(
    "Rows belonging to duplicated parent structures:",
    int(prepared["has_parent_duplicate"].sum())
)

print(
    "Unique duplicated standardized structure groups:",
    prepared.loc[
        prepared["has_standardized_duplicate"],
        "InChIKey_standardized"
    ].nunique()
)
print(
    "Unique duplicated parent structure groups:",
    prepared.loc[
        prepared["has_parent_duplicate"],
        "InChIKey_parent"
    ].nunique()
)


Rows belonging to duplicated standardized structures: 2
Rows belonging to duplicated parent structures: 22
Unique duplicated standardized structure groups: 1
Unique duplicated parent structure groups: 11


In [30]:
# Identify duplicate structure groups carrying more than one DILI-concern label.
standardized_label_counts = (
    prepared.dropna(subset=["InChIKey_standardized"])
    .groupby("InChIKey_standardized")["vDILI-Concern"]
    .nunique()
)

parent_label_counts = (
    prepared.dropna(subset=["InChIKey_parent"])
    .groupby("InChIKey_parent")["vDILI-Concern"]
    .nunique()
)

standardized_conflict_keys = set(
    standardized_label_counts[standardized_label_counts > 1].index
)

parent_conflict_keys = set(
    parent_label_counts[parent_label_counts > 1].index
)

prepared["standardized_label_conflict"] = (
    prepared["InChIKey_standardized"].isin(standardized_conflict_keys)
)

prepared["parent_label_conflict"] = (
    prepared["InChIKey_parent"].isin(parent_conflict_keys)
)

print(
    "Standardized structure groups with conflicting DILI labels:",
    len(standardized_conflict_keys)
)
print(
    "Parent structure groups with conflicting DILI labels:",
    len(parent_conflict_keys)
)

duplicate_parent_rows = prepared.loc[
    prepared["has_parent_duplicate"],
    [
        "LTKBID",
        "CompoundName",
        "vDILI-Concern",
        "SMILES_standardized",
        "SMILES_parent",
        "InChIKey_parent",
        "parent_group_size",
        "parent_label_conflict",
    ]
].sort_values(
    ["InChIKey_parent", "CompoundName"]
)

display(duplicate_parent_rows)


Standardized structure groups with conflicting DILI labels: 0
Parent structure groups with conflicting DILI labels: 5


,LTKBID,CompoundName,vDILI-Concern,SMILES_standardized,SMILES_parent,InChIKey_parent,parent_group_size,parent_label_conflict
396,LT01316,Epoetin alfa,vNo-DILI-concern,CO[C@@H]1[C@@H](O[C@@H]2O[C@H](C)[C@@H](O[C@H]...,CO[C@@H]1[C@@H](O[C@@H]2O[C@H](C)[C@@H](O[C@H]...,CJKGNDLRMLFWEX-IUJNAFBCSA-N,2,False
408,LT02330,Erythropoietin,vNo-DILI-concern,CO[C@@H]1[C@@H](O[C@@H]2O[C@H](C)[C@@H](O[C@H]...,CO[C@@H]1[C@@H](O[C@@H]2O[C@H](C)[C@@H](O[C@H]...,CJKGNDLRMLFWEX-IUJNAFBCSA-N,2,False
721,LT00946,Metoprolol succinate,Ambiguous-DILI-concern,COCCc1ccc(OCC(O)CNC(C)C)cc1.COCCc1ccc(OCC(O)CN...,COCCc1ccc(OCC(O)CNC(C)C)cc1,IUBSYMUCCVWXPE-UHFFFAOYSA-N,2,True
722,LT00309,Metoprolol tartrate,vLess-DILI-concern,COCCc1ccc(OCC(O)CNC(C)C)cc1.COCCc1ccc(OCC(O)CN...,COCCc1ccc(OCC(O)CNC(C)C)cc1,IUBSYMUCCVWXPE-UHFFFAOYSA-N,2,True
494,LT01398,Fosfomycin tromethamine,vLess-DILI-concern,C[C@@H]1O[C@@H]1P(=O)(O)O.NC(CO)(CO)CO,NC(CO)(CO)CO,LENZDBCJOHFCAS-UHFFFAOYSA-N,2,True
1155,LT00213,Tromethamine,Ambiguous-DILI-concern,NC(CO)(CO)CO,NC(CO)(CO)CO,LENZDBCJOHFCAS-UHFFFAOYSA-N,2,True
575,LT00257,Iothalamate meglumine,vNo-DILI-concern,CNC(=O)c1c(I)c(NC(C)=O)c(I)c(C(=O)O)c1I.CNC[C@...,CNC[C@H](O)[C@@H](O)[C@H](O)[C@H](O)CO,MBBZMMPHUWSWHV-BDVNFPICSA-N,2,False
1066,LT03576,Tafamidis meglumine,vNo-DILI-concern,CNC[C@H](O)[C@@H](O)[C@H](O)[C@H](O)CO.O=C(O)c...,CNC[C@H](O)[C@@H](O)[C@H](O)[C@H](O)CO,MBBZMMPHUWSWHV-BDVNFPICSA-N,2,False
344,LT00684,Divalproex sodium,vMost-DILI-concern,CCCC(CCC)C(=O)O.CCCC(CCC)C(=O)[O-].[Na+],CCCC(CCC)C(=O)O,NIJJYAXOARWZEE-UHFFFAOYSA-N,2,False
1171,LT00160,Valproic acid,vMost-DILI-concern,CCCC(CCC)C(=O)O,CCCC(CCC)C(=O)O,NIJJYAXOARWZEE-UHFFFAOYSA-N,2,False


## 7. Final Processed Structure Dataset

The final output of this notebook contains only compounds for which PubChem returned a structure and RDKit validated it successfully.

**Ambiguous DILI-concern compounds are intentionally retained.** They can be excluded later when a specific modelling experiment is defined.

No molecular descriptors, fingerprints, train/test splits, or target encodings are created here; those belong in `03_features_and_splits.ipynb`.

In [31]:
final_columns = [
    # DILIrank fields
    "LTKBID",
    "CompoundName",
    "SeverityClass",
    "LabelSection",
    "vDILI-Concern",
    "Comment",

    # PubChem / RDKit provenance
    "PubChemCID",
    "PubChemTitle",
    "pubchem_status",
    "rdkit_valid",
    "SMILES_raw",
    "InChIKey_raw",
    "SMILES_rdkit",

    # Structure composition
    "n_fragments",

    # Standardized reported form
    "standardization_ok",
    "standardization_error",
    "SMILES_standardized",
    "InChIKey_standardized",

    # Parent form for grouping / leakage checks
    "SMILES_parent",
    "InChIKey_parent",

    # Structural grouping flags
    "standardized_group_size",
    "parent_group_size",
    "has_standardized_duplicate",
    "has_parent_duplicate",
    "standardized_label_conflict",
    "parent_label_conflict",
]

final_data = prepared[final_columns].copy()

# Final consistency checks.
assert len(final_data) == n_valid
assert final_data["LTKBID"].nunique() == len(final_data)
assert final_data["pubchem_status"].eq("found").all()
assert final_data["rdkit_valid"].fillna(False).astype(bool).all()
assert final_data["vDILI-Concern"].notna().all()

print(f"Final processed structures: {len(final_data)}")
print()
print("DILI-concern distribution:")
display(
    final_data["vDILI-Concern"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print()
print(
    "Ambiguous compounds retained:",
    int(final_data["vDILI-Concern"].eq("Ambiguous-DILI-concern").sum())
)

print()
print(
    "Standardization failures:",
    int((~final_data["standardization_ok"].fillna(False)).sum())
)


Final processed structures: 1215

DILI-concern distribution:


,count
vDILI-Concern,
vNo-DILI-concern,350
vLess-DILI-concern,330
Ambiguous-DILI-concern,329
vMost-DILI-concern,206



Ambiguous compounds retained: 329

Standardization failures: 0


In [32]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_PATH = PROCESSED_DIR / "dilirank2_structures.csv"

final_data.to_csv(
    PROCESSED_PATH,
    index=False,
)

print(f"Saved processed structure dataset to: {PROCESSED_PATH}")


Saved processed structure dataset to: data/processed/dilirank2_structures.csv


## Notebook Complete

`01_data_preparation.ipynb` now produces the frozen structure-level dataset for the rest of the project.

The next notebooks can use `data/processed/dilirank2_structures.csv` without repeating PubChem retrieval or structure cleaning.

Next:

- `02_exploratory_analysis.ipynb` — explore the labels and chemical space;
- `03_features_and_splits.ipynb` — define the main No/Less/Most subset, generate RDKit descriptors and Morgan fingerprints, and create the saved random/scaffold splits.